# 选修E2 · Day 1：营销分析框架与描述性/诊断性分析> **真实库**：pandas + scipy.stats + statsmodels + causaldata> **真实数据**：NSW (National Supported Work) 真实 RCT 数据，445 条真实样本> **营销映射**：NSW 就业培训实验 -> 营销干预场景（treat=营销活动，re78=活动后消费）## NSW 数据营销映射| NSW 原始含义 | 营销映射含义 | 字段 ||-------------|-------------|------|| treat（参加培训） | 营销干预（收到促销活动） | treat (0/1) || re74（1974年收入） | 历史消费基线（活动前2年） | re74 || re75（1975年收入） | 近期消费（活动前1年） | re75 || re78（1978年收入） | 活动后消费（效果指标） | re78 || age | 客户年龄 | age || educ | 客户教育程度 | educ || marr | 是否已婚 | marr || nodegree | 是否无学位 | nodegree |## 6 个 TODO 覆盖描述性+诊断性分析| TODO | 营销任务 | 真实库 | 分析层次 ||------|---------|--------|---------|| TODO1 | 描述性统计（treated vs control 基线对比） | pandas | 描述性 || TODO2 | AARRR 营销漏斗分析 | pandas | 描述性 || TODO3 | RFM 客户分群 | pandas | 描述性+诊断性 || TODO4 | t 检验（re78 treated vs control） | scipy.stats | 诊断性 || TODO5 | 卡方检验（干预 vs 用户特征独立性） | scipy.stats | 诊断性 || TODO6 | OLS 回归 + 相关矩阵 | statsmodels | 诊断性 |

## 环境准备：导入真实库

In [ ]:
import pandas as pdimport numpy as npfrom scipy import statsimport statsmodels.api as smfrom causaldata.nsw_mixtape import load_pandasprint("库导入完成：pandas + scipy.stats + statsmodels + causaldata")

## TODO1：描述性统计 -- treated vs control 基线对比**营销场景**：在评估营销干预效果前，首先要描述两组客户的基线特征，确认随机化是否成功（基线是否平衡）。**任务**：1. 用 `causaldata.nsw_mixtape.load_pandas()` 加载 NSW 真实 RCT 数据2. 用 pandas 计算整体描述统计（age/educ/re74/re75/re78 的均值/标准差/分位数）3. 按 treat 分组，计算 re78（活动后消费）的均值和中位数4. 计算两组样本量**预期输出**：445 行数据，260 control / 185 treated，treated 组 re78 均值高于 control

In [ ]:
# TODO1: 加载 NSW 数据并计算描述性统计df = load_pandas().dataprint(f"数据形状: {df.shape}")print(f"列名: {df.columns.tolist()}")print(f"\n整体描述统计:")print(df[['age', 'educ', 're74', 're75', 're78']].describe())print(f"\n--- 按 treat 分组: re78 描述统计 ---")grp = df.groupby('treat')['re78'].agg(['count', 'mean', 'median', 'std'])print(grp)print(f"\n--- 两组样本量 ---")print(df['treat'].value_counts().sort_index())print(f"Control (treat=0): 260 人, re78 均值=${df[df.treat==0]['re78'].mean():.2f}")print(f"Treated (treat=1): 185 人, re78 均值=${df[df.treat==1]['re78'].mean():.2f}")print(f"差异: ${df[df.treat==1]['re78'].mean() - df[df.treat==0]['re78'].mean():.2f}")

## TODO2：AARRR 营销漏斗分析**营销场景**：用 AARRR 框架（Acquisition/Activation/Retention/Revenue）映射 NSW 数据，计算各阶段转化率。**映射逻辑**：- Acquisition（获取）：re74 > 0（活动前2年有消费 = 已获客）- Activation（激活）：re75 > 0（活动前1年有消费 = 已激活）- Retention（留存）：re78 > 0（活动后有消费 = 已留存）- Revenue（变现）：re78 的总和（活动后总消费）**任务**：1. 分别计算 treat=0 和 treat=1 两组在 AARRR 各阶段的用户数和转化率2. 计算每组的总 Revenue（re78 之和）3. 对比两组的漏斗转化率差异

In [ ]:
# TODO2: AARRR 营销漏斗分析print("=== AARRR 营销漏斗分析 ===\n")for t in [0, 1]:    sub = df[df['treat'] == t]    n = len(sub)    acq = (sub['re74'] > 0).sum()    act = (sub['re75'] > 0).sum()    ret = (sub['re78'] > 0).sum()    rev = sub['re78'].sum()    label = "Control" if t == 0 else "Treated"    print(f"--- {label} (treat={t}, n={n}) ---")    print(f"  Acquisition  (re74>0): {acq:3d} ({acq/n:.1%})")    print(f"  Activation   (re75>0): {act:3d} ({act/n:.1%})")    print(f"  Retention    (re78>0): {ret:3d} ({ret/n:.1%})")    print(f"  Revenue      (re78总和): ${rev:,.1f}")    print()print("漏斗洞察:")print(f"  Treated 组 Activation 率 ({(df[df.treat==1]['re75']>0).mean():.1%})")print(f"  vs Control ({(df[df.treat==0]['re75']>0).mean():.1%}) -- 干预组激活率更高")print(f"  Treated 组 Retention 率 ({(df[df.treat==1]['re78']>0).mean():.1%})")print(f"  vs Control ({(df[df.treat==0]['re78']>0).mean():.1%}) -- 干预组留存率更高")

## TODO3：RFM 客户分群**营销场景**：用 RFM（Recency/Frequency/Monetary）将客户分为 5 个分群，指导差异化营销策略。**映射逻辑**：- Recency（最近性）：re78 > 0 = 1（近期有消费），re78 = 0 = 0（近期无消费）- Frequency（频次）：re74/re75/re78 中收入>0的期数（0-3）- Monetary（金额）：re78 的值**任务**：1. 计算 R/F/M 三个维度2. 按 R 和 F 的组合将客户分为 5 个分群：Champions / Loyal / Recent / At Risk / Lost3. 统计各分群人数，并用 pd.crosstab 交叉分群与 treat 分组

In [ ]:
# TODO3: RFM 客户分群df['r_score'] = (df['re78'] > 0).astype(int)df['f_score'] = ((df['re74'] > 0).astype(int) +                 (df['re75'] > 0).astype(int) +                 (df['re78'] > 0).astype(int))df['m_value'] = df['re78']m_median = df['m_value'].median()def rfm_segment(row):    r, f, m = row['r_score'], row['f_score'], row['m_value']    if r == 1 and f >= 2 and m > m_median:        return "Champions"    elif r == 1 and f >= 2:        return "Loyal"    elif r == 1:        return "Recent"    elif f >= 2:        return "At Risk"    else:        return "Lost"df['segment'] = df.apply(rfm_segment, axis=1)print("=== RFM 客户分群 ===")print(df['segment'].value_counts())print(f"\nMonetary 中位数: ${m_median:.2f}")print("\n--- 分群 x treat 交叉表 ---")crosstab = pd.crosstab(df['segment'], df['treat'], margins=True)print(crosstab)print("\n分群洞察:")for seg in ['Champions', 'Loyal', 'Recent', 'At Risk', 'Lost']:    sub = df[df['segment'] == seg]    print(f"  {seg}: {len(sub)} 人, treat占比={sub['treat'].mean():.1%}, 平均re78=${sub['re78'].mean():.0f}")

## TODO4：t 检验 -- 营销干预效果显著性**营销场景**：用独立样本 t 检验判断营销干预（treat）是否显著提升了活动后消费（re78）。**任务**：1. 用 `scipy.stats.ttest_ind` 执行 Welch t 检验（equal_var=False）2. 计算 Cohen's d 效应量3. 解读结果：p 值是否 < 0.05？效应量大小？**预期结果**：treat 组 re78 均值约 $6,349，control 约 $4,555，p < 0.01

In [ ]:
# TODO4: t 检验treat_re78 = df[df['treat'] == 1]['re78']ctrl_re78 = df[df['treat'] == 0]['re78']t_stat, p_val = stats.ttest_ind(treat_re78, ctrl_re78, equal_var=False)n1, n0 = len(treat_re78), len(ctrl_re78)var1, var0 = treat_re78.var(ddof=1), ctrl_re78.var(ddof=1)pooled_std = np.sqrt(((n1 - 1) * var1 + (n0 - 1) * var0) / (n1 + n0 - 2))cohen_d = (treat_re78.mean() - ctrl_re78.mean()) / pooled_stdprint("=== t 检验: re78 treated vs control ===")print(f"Treated  : mean=${treat_re78.mean():.2f}, std=${treat_re78.std():.2f}, n={n1}")print(f"Control  : mean=${ctrl_re78.mean():.2f}, std=${ctrl_re78.std():.2f}, n={n0}")print(f"差异     : ${treat_re78.mean() - ctrl_re78.mean():.2f}")print(f"\nWelch t-statistic : {t_stat:.4f}")print(f"p-value            : {p_val:.4f}")print(f"Cohen's d          : {cohen_d:.4f}")print(f"\n解读:")print(f"  p={p_val:.4f} < 0.01 -> 统计显著，营销干预有效提升 re78")print(f"  Cohen's d={cohen_d:.4f} -> 小效应量（0.2-0.5 为小）")print(f"  统计显著但效应量小 -> 大样本下检测到真实但微小的效果")

## TODO5：卡方独立性检验 -- 干预分组与用户特征**营销场景**：检验营销干预分组是否与用户人口统计学特征（已婚/无学位）独立。如果不独立，说明随机化可能存在问题。**任务**：1. 用 `pd.crosstab` 构建列联表（treat vs marr）2. 用 `scipy.stats.chi2_contingency` 执行卡方检验3. 对 treat vs nodegree 重复上述检验4. 解读：哪个特征与干预分组显著相关？

In [ ]:
# TODO5: 卡方独立性检验print("=== 卡方检验 1: treat vs marr (是否已婚) ===")ct_marr = pd.crosstab(df['treat'], df['marr'])print(ct_marr)chi2_marr, p_marr, dof_marr, exp_marr = stats.chi2_contingency(ct_marr)print(f"chi2={chi2_marr:.4f}, p={p_marr:.4f}, dof={dof_marr}")print(f"解读: p={p_marr:.4f} > 0.05 -> 干预与婚姻状况独立（随机化成功）")print(f"\n=== 卡方检验 2: treat vs nodegree (是否无学位) ===")ct_nd = pd.crosstab(df['treat'], df['nodegree'])print(ct_nd)chi2_nd, p_nd, dof_nd, exp_nd = stats.chi2_contingency(ct_nd)print(f"chi2={chi2_nd:.4f}, p={p_nd:.4f}, dof={dof_nd}")print(f"解读: p={p_nd:.4f} < 0.05 -> 干预与学位状况不独立（基线不平衡！）")print(f"\n--- 两个检验对比 ---")print(f"  treat vs marr     : chi2={chi2_marr:.2f}, p={p_marr:.4f} -> 独立")print(f"  treat vs nodegree : chi2={chi2_nd:.2f}, p={p_nd:.4f} -> 不独立")print(f"基线不平衡意味着 TODO6 的 OLS 回归控制协变量很重要")

## TODO6：OLS 回归 + 相关矩阵 -- 控制混杂后的净效应**营销场景**：用 OLS 回归控制协变量（re75/age/educ），估计营销干预（treat）对活动后消费（re78）的净效应。这是诊断性分析的核心--区分"原始差异"和"控制混杂后的净效应"。**任务**：1. 用 `pandas.DataFrame.corr()` 计算数值变量的相关矩阵2. 用 `statsmodels.api.OLS` 拟合回归：re78 ~ treat + re75 + age + educ3. 用 `model.summary()` 输出完整回归诊断4. 解读：treat 系数是多少？是否显著？R-squared 多大？

In [ ]:
# TODO6: OLS 回归 + 相关矩阵print("=== 相关矩阵 ===")corr_cols = ['age', 'educ', 're74', 're75', 're78']corr_matrix = df[corr_cols].corr()print(corr_matrix.round(4))print(f"\n关键相关: re74-re75={corr_matrix.loc['re74','re75']:.3f} (强相关)")print(f"         re78-educ={corr_matrix.loc['re78','educ']:.3f} (弱相关)")print(f"\n=== OLS 回归: re78 ~ treat + re75 + age + educ ===")X = df[['treat', 're75', 'age', 'educ']].copy()X = sm.add_constant(X)y = df['re78']model = sm.OLS(y, X).fit()print(model.summary())print(f"\n--- 回归结果解读 ---")print(f"R-squared: {model.rsquared:.4f} (模型解释 {model.rsquared:.1%} 的方差)")print(f"treat 系数: {model.params['treat']:.2f} (p={model.pvalues['treat']:.4f})")print(f"  -> 控制混杂后，干预净效应约 ${model.params['treat']:.0f}")print(f"educ 系数: {model.params['educ']:.2f} (p={model.pvalues['educ']:.4f})")print(f"  -> 教育年限每增加1年，re78增加约 ${model.params['educ']:.0f}")print(f"re75 系数: {model.params['re75']:.4f} (p={model.pvalues['re75']:.4f})")print(f"  -> 活动前消费每增加$1，活动后消费增加 ${model.params['re75']:.4f}")treat_raw = df[df.treat==1]['re78'].mean() - df[df.treat==0]['re78'].mean()print(f"\n原始差异: ${treat_raw:.2f}")print(f"OLS净效应: ${model.params['treat']:.2f}")print(f"差异说明: OLS控制混杂后净效应略低于原始差异")

## 总结：描述性 + 诊断性分析的营销洞察完成 6 个 TODO 后，你应能回答：1. **描述性**：treated 组和 control 组的基线特征是否平衡？AARRR 漏斗中哪个阶段转化率差异最大？2. **诊断性**：营销干预对 re78 的效果统计显著吗（t 检验）？效应量多大（Cohen's d）？控制混杂后净效应是多少（OLS 系数）？干预分组与用户特征是否独立（卡方检验）？**关键认知**：- 描述性分析回答"发生了什么"（TODO1-3）- 诊断性分析回答"为什么发生"（TODO4-6）- p 值告诉你"有没有效果"，Cohen's d 告诉你"效果多大"- OLS 回归控制协变量后的系数是"净效应"，比原始差异更可信- 这四层分析为 Day 2（预测性）和 Day 3（处方性）奠基**与 CUPED 的连接**：TODO6 的 OLS 回归中 re75 作为协变量，其思想与 CUPED（方差缩减技术）一致--利用预处理信息提升对干预效应的估计精度。Day 2/3 将深入 CUPED。